# ARC-AGI-2 Semantic Quotient Search v0.1

Research baseline: generate a small symbolic hypothesis set, collapse programs that are **observationally equivalent on all ARC demonstrations**, keep the simplest representative, rank surviving behaviors, validate on public evaluation tasks, and write Kaggle `submission.json`.

For a finite hypothesis space \(H\), uniform Hartley uncertainty is \(\log_2|H|\). Each demonstration removes inconsistent behaviors. "Semantic" here is deliberately narrow: observational/task equivalence under the demonstrations, not universal meaning.

**Kaggle setup:** attach the **ARC Prize 2026 - ARC-AGI-2** competition as notebook input. Importing this notebook from GitHub does not automatically attach competition data. The Paper Track itself has no dataset.

In [ ]:
from __future__ import annotations
import json, math, time
from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict
from typing import Callable
import numpy as np, pandas as pd

REQUIRED = {
    "arc-agi_training_challenges.json",
    "arc-agi_evaluation_challenges.json",
    "arc-agi_evaluation_solutions.json",
    "arc-agi_test_challenges.json",
}

def data_dir():
    roots = [
        Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2"),
        Path("/kaggle/input/arc-prize-2026-arc-agi-2"),
    ]
    for p in roots:
        if p.exists() and REQUIRED.issubset({x.name for x in p.iterdir()}):
            return p

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for marker in kaggle_input.rglob("arc-agi_evaluation_challenges.json"):
            p = marker.parent
            names = {x.name for x in p.iterdir()}
            if {"arc-agi_evaluation_solutions.json", "arc-agi_test_challenges.json"}.issubset(names):
                return p

    attached = []
    if kaggle_input.exists():
        attached = sorted(str(p) for p in kaggle_input.iterdir())

    raise FileNotFoundError(
        "\nARC-AGI-2 competition data is not attached to this notebook.\n\n"
        "Kaggle fix:\n"
        "  1. Open this notebook in Kaggle.\n"
        "  2. In the right sidebar choose Add Input (or + Add Input).\n"
        "  3. Search for: ARC Prize 2026 - ARC-AGI-2\n"
        "  4. Add the competition data and accept/join the competition rules if Kaggle asks.\n"
        "  5. Restart/run this notebook.\n\n"
        "Important: the ARC Prize 2026 Paper Track has no dataset, so attaching/opening "
        "the notebook only there is not enough.\n\n"
        f"Currently visible /kaggle/input entries: {attached or '[none]'}"
    )

D = data_dir()
print("ARC data directory:", D)
print("Files:", sorted(p.name for p in D.glob("*.json")))

def load(name):
    with open(D / name) as f:
        return json.load(f)

EV = load("arc-agi_evaluation_challenges.json")
ES = load("arc-agi_evaluation_solutions.json")
TE = load("arc-agi_test_challenges.json")
print("evaluation tasks:", len(EV), "| test tasks:", len(TE))

def A(g): return np.asarray(g, dtype=np.int8)
def key(x): return tuple(map(tuple, np.asarray(x).tolist()))

def bg(x):
    v, c = np.unique(x, return_counts=True)
    return int(v[np.argmax(c)])

def crop(x):
    m = x != bg(x)
    pts = np.argwhere(m)
    if not len(pts): return None
    a, b = pts.min(0), pts.max(0)
    return x[a[0]:b[0]+1, a[1]:b[1]+1].copy()

@dataclass
class P:
    name: str
    cost: float
    fn: Callable
    def __call__(self, g):
        try:
            y = self.fn(A(g))
            if y is None: return None
            y = A(y)
            if y.ndim != 2 or not y.size or max(y.shape) > 30: return None
            return y
        except Exception:
            return None

def prim():
    return [
        P("id", 1, lambda x: x.copy()),
        P("r90", 2, lambda x: np.rot90(x, 1).copy()),
        P("r180", 2, lambda x: np.rot90(x, 2).copy()),
        P("r270", 2, lambda x: np.rot90(x, 3).copy()),
        P("flr", 2, lambda x: np.fliplr(x).copy()),
        P("fud", 2, lambda x: np.flipud(x).copy()),
        P("T", 2, lambda x: x.T.copy()),
        P("crop", 3, crop),
    ]

def comp(p, q):
    def f(x):
        y = p(x)
        return None if y is None else q(y)
    return P(q.name + "@" + p.name, p.cost + q.cost + .5, f)

def cmap(train, p):
    m = {}
    changed = False
    for z in train:
        u = p(z["input"]); v = A(z["output"])
        if u is None or u.shape != v.shape: return None
        for a, b in zip(u.ravel(), v.ravel()):
            a, b = int(a), int(b)
            if a in m and m[a] != b: return None
            m[a] = b
            changed |= a != b
    if not changed: return None

    def f(x):
        u = p(x)
        if u is None: return None
        v = u.copy()
        for a, b in m.items():
            v[u == a] = b
        return v

    return P(p.name + "+map" + str(sorted(m.items())), p.cost + 1 + .2 * len(m), f)

def hypotheses(task):
    b = prim()
    ps = b + [comp(p, q) for p in b for q in b]
    ps += [z for z in (cmap(task["train"], p) for p in list(ps)) if z]
    out = {}
    for p in ps:
        if p.name not in out or p.cost < out[p.name].cost:
            out[p.name] = p
    return list(out.values())

def sig(p, ins):
    return tuple(None if (y := p(x)) is None else key(y) for x in ins)

def quotient(ps, ins):
    q = defaultdict(list)
    for p in ps:
        q[sig(p, ins)].append(p)
    return [min(v, key=lambda p: (p.cost, p.name)) for v in q.values()]

def score(p, train):
    exact, pix = 0, []
    for z in train:
        y = p(z["input"]); t = A(z["output"])
        ok = y is not None and y.shape == t.shape
        exact += int(ok and np.array_equal(y, t))
        pix.append(float(np.mean(y == t)) if ok else 0.)
    return exact, float(np.mean(pix)), p.cost

def solve(task):
    start = time.perf_counter()
    raw = hypotheses(task)
    ins = [z["input"] for z in task["train"]]
    qs = quotient(raw, ins)
    ranked = sorted(
        qs,
        key=lambda p: (
            -score(p, task["train"])[0],
            -score(p, task["train"])[1],
            p.cost, p.name
        )
    )
    exact_survivors = sum(
        score(p, task["train"])[0] == len(task["train"]) for p in qs
    )

    ans = []
    for z in task["test"]:
        uniq = []
        seen = set()
        for p in ranked:
            y = p(z["input"])
            if y is not None and key(y) not in seen:
                seen.add(key(y))
                uniq.append((y, p.name))
            if len(uniq) == 2: break
        if not uniq:
            uniq = [(A(z["input"]), "fallback")]
        if len(uniq) == 1:
            uniq.append(uniq[0])
        ans.append({
            "attempt_1": uniq[0][0].astype(int).tolist(),
            "attempt_2": uniq[1][0].astype(int).tolist(),
        })

    d = {
        "raw": len(raw),
        "classes": len(qs),
        "ratio": len(raw) / max(1, len(qs)),
        "survivors": exact_survivors,
        "hartley": 0 if len(qs) <= 1 else math.log2(len(qs)),
        "seconds": time.perf_counter() - start,
    }
    return ans, d


In [ ]:
# Public evaluation. Set LIMIT=50 for a quick run or None for all tasks.
LIMIT = None
rows = []
correct = total = 0

for i, (tid, task) in enumerate(EV.items()):
    if LIMIT is not None and i >= LIMIT:
        break
    pred, d = solve(task)
    truth = ES[tid]
    hit = 0
    for p, y in zip(pred, truth):
        y = A(y); a = A(p["attempt_1"]); b = A(p["attempt_2"])
        ok = ((a.shape == y.shape and np.array_equal(a, y)) or
              (b.shape == y.shape and np.array_equal(b, y)))
        correct += int(ok); total += 1; hit += int(ok)
    rows.append({"task": tid, "correct": hit, "outputs": len(truth), **d})

df = pd.DataFrame(rows)
print("exact output accuracy =", correct / max(1, total))
display(df.head())
display(df.describe())
df.to_csv("semantic_quotient_eval.csv", index=False)


In [ ]:
# Kaggle submission: exactly two attempts for every hidden test output.
submission = {}
diag = []

for tid, task in TE.items():
    pred, d = solve(task)
    submission[tid] = pred
    diag.append({"task": tid, **d})

assert set(submission) == set(TE)
for tid, v in submission.items():
    assert len(v) == len(TE[tid]["test"])
    assert all(set(x) == {"attempt_1", "attempt_2"} for x in v)

with open("submission.json", "w") as f:
    json.dump(submission, f)

pd.DataFrame(diag).to_csv("semantic_quotient_test.csv", index=False)
print("wrote submission.json for", len(submission), "tasks")


## Interpretation and next experiment

v0.1 tests a falsifiable claim: **does collapsing task-indistinguishable candidate behaviors reduce search burden without reducing ARC accuracy?**

Next: quotient *during* program expansion, then add object/relation primitives, non-uniform priors, and information-gain-per-compute scheduling.